# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [23]:
df = pd.read_csv('data/AviationData.csv', encoding='ISO-8859-1', low_memory=False)
# Initial Inspection
print(df.info())
print(df.isna().sum())
display(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [24]:
# 1. Convert Event.Date to datetime
df['Event.Date'] = pd.to_datetime(df['Event.Date'])

# 2. Filter for data from 1983 onwards
df_modern = df[df['Event.Date'].dt.year >= 1983].copy()

# 3. Filter for Airplanes only (excluding helicopters, gliders, etc. if present)
df_modern = df_modern[df_modern['Aircraft.Category'] == 'Airplane']

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

In [25]:
print(df.columns.tolist())

['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date']


**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [26]:
# List of injury columns
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

# Fill NaNs with 0 for calculation
for col in injury_cols:
    df_modern[col] = df_modern[col].fillna(0)

# Create Total.Passengers and Injury.Rate
df_modern['Total.Passengers'] = df_modern[injury_cols].sum(axis=1)

# Calculate rate (prevent division by zero)
df_modern['Fatal_Serious_Rate'] = (df_modern['Total.Fatal.Injuries'] + df_modern['Total.Serious.Injuries']) / df_modern['Total.Passengers']
df_modern['Fatal_Serious_Rate'] = df_modern['Total.Serious.Injuries'].fillna(0)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [27]:
# Tracking total destruction
df_modern['Is_Destroyed'] = df_modern['Aircraft.damage'].apply(
    lambda x: 1 if str(x).lower() == 'destroyed' else 0
)

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [28]:
# Standardize Make names
df_modern['Make'] = df_modern['Make'].str.upper().str.strip()

# Filtering for frequency (Threshold = 50)
make_counts = df_modern['Make'].value_counts()
robust_makes = make_counts[make_counts >= 50].index
df_final = df_modern[df_modern['Make'].isin(robust_makes)].copy()

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [29]:
# Create Unique Model Identifier
df_final['Model'] = df_final['Model'].str.upper().str.strip()
df_final.dropna(subset=['Model'], inplace=True)
df_final['Unique_Model_ID'] = df_final['Make'] + " " + df_final['Model']

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [30]:
# Simple imputation and standardization
df_final['Weather.Condition'] = df_final['Weather.Condition'].fillna('Unknown').str.upper()
df_final['Broad.phase.of.flight'] = df_final['Broad.phase.of.flight'].fillna('Unknown').str.upper()

# Drop columns with > 80% missing data (e.g., Latitude/Longitude in older records)
limit = len(df_final) * 0.8
df_final = df_final.dropna(thresh=limit, axis=1)

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [31]:
df_final.to_csv('Aviation_Data_Cleaned.csv', index=False)
print(f"Cleaning Complete. {len(df_final)} records ready for recommendation analysis.")

Cleaning Complete. 18091 records ready for recommendation analysis.


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized